## Description
This script organizes a list of menus and submenus from the "Activity Name" column in the CAR data into a hierarchical structure (Tier1–Tier3) to support hierarchical filtering and analysis in Power BI. It also groups specific single-level menus under a new Tier1 category called "Miscellaneous" to make filters easier to sort through. The resulting CSV file should be uploaded into Power BI and linked with your CAR data files via the "Activity Name" column to enable hierarchical filtering (when you do this, make sure the cross-filtering setting is set to "Both").

Note that this is an updated version of my previous code for hierarchical filtering, which includes a cleaner and more user-friendly setup for the filter box in Power BI. The CSV file created by this code is something that I manually created based on the menu maps to better meet business needs.

In [1]:
import pandas as pd
import numpy as np
from io import StringIO

# --- Step 1: I pasted the full table which I manually created as a raw multiline string (tab-separated) ---
raw = """Activity_Name	Tier	Tier1	Tier2	Tier3
CCB	1	Miscellaneous	CCB	
ClosedQueueMenu	1	Closed Queue Menu		
ClinicVoicemailTransfer	2	Closed Queue Menu	Clinic Voicemail Transfer	
DisconnectContact	1	Disconnect Contact		
DisconnectContact1	2	Disconnect Contact	Disconnect Contact 1	
DisconnectContact2	2	Disconnect Contact	Disconnect Contact 2	
DisconnectCallbackContact	2	Disconnect Contact	Disconnect Callback Contact	
FarmworkerMainMenu	1	Farmworker Main Menu		
FrontDeskTransfer	1	Front Desk Transfer		
FrontDeskTransfer1	2	Front Desk Transfer	Front Desk Transfer 1	
FrontDeskTransfer2	2	Front Desk Transfer	Front Desk Transfer 2	
FrontDeskTransfer3	2	Front Desk Transfer	Front Desk Transfer 3	
IntakePreQueueMessage1	1	Miscellaneous	Intake Pre-QueueMessage 1	
PreQueueMessage2	2	Miscellaneous	Intake Pre-QueueMessage 1	Pre-QueueMessage 2
LegalServerScreenPop	2	Miscellaneous	Intake Pre-QueueMessage 1	Legal Server Screen-Pop
ScreenPopProcessComplete	2	Miscellaneous	Intake Pre-QueueMessage 1	Screen-Pop: Process Complete
PlayMOH300s	2	Miscellaneous	Intake Pre-QueueMessage 1	Play Music
QueueMenu1	2	Miscellaneous	Intake Pre-QueueMessage 1	Queue Menu 1
LegalMenu2	2	Legal Issues		
ADAPTQueue	3	Legal Issues	ADAPT Queue	
ADAPTSPQueue	3	Legal Issues	ADAPT Queue	
CriminalRecordsVoicemailTransfer	3	Legal Issues	Criminal Records Voicemail Transfer	
ConsumerSPQueue	3	Legal Issues	Consumer Queue	
ConsumerQueue	3	Legal Issues	Consumer Queue	
HousingMenu	3	Legal Issues	Housing Menu	
TenantMenu	5	Legal Issues	Housing Menu	TenantMenu
TenantDeterrenceMenu	5	Legal Issues	Housing Menu	TenantDeterrenceMenu
HousingQueue	4	Legal Issues	Housing Menu	Housing Queue
HousingSPQueue	4	Legal Issues	Housing Menu	Housing Queue
EmploymentMenu	3	Legal Issues	Employment Menu	
EmploymentQueue	4	Legal Issues	Employment Menu	Employment Queue
EmploymentSPQueue	4	Legal Issues	Employment Menu	Employment Queue
WorkersCompMenu	4	Legal Issues	Employment Menu	Workers Comp Menu
ImmigrationMenu	3	Legal Issues	Employment Menu	
ImmigrationOtherMenu	4	Legal Issues	Employment Menu	Immigration Other Menu
ImmigrationSPQueue	4	Legal Issues	Employment Menu	Immigration Queue
ImmigrationQueue	4	Legal Issues	Employment Menu	Immigration Queue
MigrantVoicemailTransfer	4	Legal Issues	Employment Menu	Migrant Voicemail Transfer
TraffickingVoicemailTransfer	4	Legal Issues	Employment Menu	Trafficking Voicemail Transfer
FamilyMenu	3	Legal Issues	Family Menu	
SimpleDivorceMenu	5	Legal Issues	Family Menu	Simple Divorce Menu
ChildSupportMenu	5	Legal Issues	Family Menu	Child Support Menu
FamilyQueue	4	Legal Issues	Family Menu	Family Queue
EducationQueue	4	Legal Issues	Family Menu	Education Queue
EducationSPQueue	4	Legal Issues	Family Menu	Education Queue
FamilySPQueue	4	Legal Issues	Family Menu	Family Queue
OtherLegalMenu	3	Legal Issues	Other Legal Menu	
OtherLegalCriminalCaseMenu	4	Legal Issues	Other Legal Menu	Other Legal Criminal Case Menu
OtherLegalPersonalInjuryMenu	4	Legal Issues	Other Legal Menu	Other Legal Personal Injury Menu
OtherLegalOtherMenu	4	Legal Issues	Other Legal Menu	Other Legal Other Menu
BenefitsMenu	3	Legal Issues	Benefits Menu	
BenefitsQueue	4	Legal Issues	Benefits Menu	Benefits Queue
BenefitsSPQueue	4	Legal Issues	Benefits Menu	Benefits Queue
VeteransBenefitsVoicemailTransfer	4	Legal Issues	Benefits Menu	Veterans Benefits Voicemai lTransfer
HIVMenu	3	Legal Issues	HIV Menu	
HIVVoicemailTransfer	4	Legal Issues	HIV Menu	HIV Voicemail Transfer
MainMenu	1	Main Menu		
AddressFaxHoursMenu	2	Main Menu	Address Fax Hours Menu	
ComplimentOrComplaintMenu	2	Main Menu	Compliment Or Complaint Menu	
AppointmentMenu	2	Main Menu	Appointment Menu	
HolidayPrompt	2	Main Menu	Holiday Prompt	
LanguageSelectionMenu	2	Main Menu	Language Selection Menu	
StaffDirectoryEnglishTransfer	2	Main Menu	Staff Directory English Transfer	
StaffDirectorySpanishTransfer	2	Main Menu	Staff Directory  Spanish Transfer	
SetCallerID	2	Miscellaneous	Set Caller ID	
HelpWithLegalorOtherReasonMenu	2	Miscellaneous	Help With Legal or Other Reason Menu	
ClosedMenu	2	Miscellaneous	Closed Menu	
ReadANI	2	Miscellaneous	ReadANI	
CallbackRetry	2	Miscellaneous	CallbackRetry	
ConfirmCallbackNumber	2	Miscellaneous	Confirm Callback Number	
CollectCallbackNumber	2	Miscellaneous	Collect Callback Number	
TransferToSafeHaven	2	Miscellaneous	Transfer To Safe Haven	
SeniorsMenu	1	Seniors Menu		
SeniorNotCookCoMenu	3	Seniors Menu	Seniors: Not Cook Count Menu (exit)	
SuburbanSeniorsMenu	3	Seniors Menu	Suburban Seniors Menu	
SubSeniorHomeownerQueue	4	Seniors Menu	Suburban Seniors Menu	SS Homeowner Queue
SubSeniorHomeownerSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Homeowner Queue
SubSeniorOtherQueue	4	Seniors Menu	Suburban Seniors Menu	SS Other Queue
SubSeniorPreQueueMessage1_1	4	Seniors Menu	Suburban Seniors Menu	SS Pre-Queue Message
SubSeniorConsumerQueue	4	Seniors Menu	Suburban Seniors Menu	SubSeniorConsumerQueue
SubSeniorTenantQueue	4	Seniors Menu	Suburban Seniors Menu	SubSeniorTenantQueue
SubSeniorPreQueueMessage1_2	4	Seniors Menu	Suburban Seniors Menu	SS Pre-Queue Message
BenefitsSubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
SubSeniorFamilyQueue	4	Seniors Menu	Suburban Seniors Menu	SS Family Queue
SubSeniorBenefitsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
SubSeniorTenantSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Tenant Queue
SubSeniorEmploymentQueue	4	Seniors Menu	Suburban Seniors Menu	SS EmploymentQueue
SubSeniorConsumerSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
HousingSubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Houring Queue
EmploymentSubSeniorsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS EmploymentQueue
OtherSubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Other Queue
FamilySubSeniorsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Family Queue
HousingSubSeniorsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Houring Queue
BenefitsSubSeniorsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
ConsumerSubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
EmploymentSubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS EmploymentQueue
OtherSubSeniorsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Other Queue
FamilySubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Family Queue
HomeownerSubSeniorsQueue	4	Seniors Menu	Suburban Seniors Menu	SS Homeowner Queue
ConsumerSubSeniorsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
SubSeniorEmploymentSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS EmploymentQueue
SubSeniorOtherSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Other Queue
SubSeniorFamilySPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Family Queue
SubSeniorBenefitsSPQueue	4	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
SeniorsADAPTMenu	4	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
SubSeniorADAPTQueue	5	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
ADAPTSubSeniorsQueue	5	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
SubSeniorADAPTSPQueue	5	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
ADAPTSubSeniorsSPQueue	5	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue"""

# --- Step 2: Read the string into a DataFrame using pandas ---
df = pd.read_csv(StringIO(raw), sep="\t", dtype=str)

# --- Step 3: Normalize columns and datatypes ---
for col in ["Activity_Name", "Tier", "Tier1", "Tier2", "Tier3"]:
    if col not in df.columns:
        df[col] = np.nan

# Convert Tier to integer
df["Tier"] = pd.to_numeric(df["Tier"], errors="coerce").astype("Int64")

# Replace empty strings with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# --- Step 4: Save to CSV for reproducible Power BI load ---
df.to_csv("Hierarchical_Tiers.csv", index=False)

# --- Step 5: Preview ---
display(df.head(10))

,Activity_Name,Tier,Tier1,Tier2,Tier3
0,CCB,1,Miscellaneous,CCB,NaN
1,ClosedQueueMenu,1,Closed Queue Menu,NaN,NaN
2,ClinicVoicemailTransfer,2,Closed Queue Menu,Clinic Voicemail Transfer,NaN
3,DisconnectContact,1,Disconnect Contact,NaN,NaN
4,DisconnectContact1,2,Disconnect Contact,Disconnect Contact 1,NaN
5,DisconnectContact2,2,Disconnect Contact,Disconnect Contact 2,NaN
6,DisconnectCallbackContact,2,Disconnect Contact,Disconnect Callback Contact,NaN
7,FarmworkerMainMenu,1,Farmworker Main Menu,NaN,NaN
8,FrontDeskTransfer,1,Front Desk Transfer,NaN,NaN
9,FrontDeskTransfer1,2,Front Desk Transfer,Front Desk Transfer 1,NaN
